In [1]:
import math
import os
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import datasets
import einops
import numpy as np
import torch as t
import torch.nn as nn
import wandb
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table
from torch import Tensor
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from transformer_lens import HookedTransformer
from transformer_lens.utils import gelu_new, tokenize_and_concatenate
from transformers import GPT2TokenizerFast

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")


In [2]:
gpt = HookedTransformer.from_pretrained("gpt2-small")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer


In [5]:
vocab = sorted(list(gpt.tokenizer.vocab.items()), key=lambda x: x[1])

In [17]:
gpt.tokenizer.decode([26])
gpt.to_str_tokens

<bound method HookedTransformer.to_str_tokens of HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookP

In [31]:
# gpt.tokenizer.vocab["Ġthe"]
text = "The cat sat on the mat."

tokens = gpt.to_tokens(text)
print(tokens)
print(tokens.shape)

tensor([[50256,   464,  3797,  3332,   319,   262,  2603,    13]],
       device='mps:0')
torch.Size([1, 8])


In [29]:
logits, cache = gpt.run_with_cache(tokens) 
print(logits.shape)

torch.Size([1, 8, 50257])


In [33]:
# # logits.shape
# print("embed \t\t", cache['hook_embed'].shape)
# print("pos_embed \t", cache['hook_pos_embed'].shape)
# print()
# print("resid_pre \t", cache['blocks.0.hook_resid_pre'].shape)
# print("scale \t\t", cache['blocks.0.ln1.hook_scale'].shape)
# print("normalized \t", cache['blocks.0.ln1.hook_normalized'].shape)   
# print()
# print("q \t\t", cache['blocks.0.attn.hook_q'].shape)
# print("k \t\t", cache['blocks.0.attn.hook_k'].shape)
# print("v \t\t", cache['blocks.0.attn.hook_v'].shape)
# print()
# print("attn_scores \t", cache['blocks.0.attn.hook_attn_scores'].shape)
# print("pattern \t", cache['blocks.0.attn.hook_pattern'].shape)
# print("z \t\t", cache['blocks.0.attn.hook_z'].shape)
# print()
# print("attn_out \t", cache['blocks.0.hook_attn_out'].shape)
# print()
# print("resid_mid \t", cache['blocks.0.hook_resid_mid'].shape)
# print()
# print("scale \t\t", cache['blocks.0.ln2.hook_scale'].shape)
# print("normalized \t", cache['blocks.0.ln2.hook_normalized'].shape)
# print()
# print("pre \t\t", cache['blocks.0.mlp.hook_pre'].shape)
# print("post \t\t", cache['blocks.0.mlp.hook_post'].shape)
# print("mlp_out \t", cache['blocks.0.hook_mlp_out'].shape)
# print()
# print("resid_post \t", cache['blocks.0.hook_resid_post'].shape)
# print()
# print("ln_final.scale \t", cache['ln_final.hook_scale'].shape)
# print("ln_final.normalized \t", cache['ln_final.hook_normalized'].shape)